# Appendix B Companion Notebook: Python Basics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/appendices/Appendix_B_Python_Basics.ipynb)

This notebook accompanies Appendix B of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook turns Appendix B into guided Python practice for readers who are new to programming. A synthetic retail sales example connects variables, collections, NumPy arrays, pandas DataFrames, visualization, file handling, and responsible analytical workflow design.

## How to use this notebook

Run the cells from top to bottom. Read the explanation before each code block, inspect the output, and change one element at a time while learning. The examples create their own synthetic files, so no upload is required.

Before sharing the notebook, restart the runtime and run all cells. A clean run confirms that the analysis does not depend on hidden variables or an accidental execution order.

## Why this matters (business framing)

Python is useful because it can keep an analytical process in one reproducible workflow. The same notebook can document a business question, import data, correct quality problems, calculate metrics, create charts, and save results for review. The value is not the syntax alone. The value is a transparent chain from question to evidence to decision.

## Agenda

1. Setup and reproducibility
2. Python in the business analytics workflow
3. Practical workspace and package choices
4. Variables, operators, conditions, loops, and functions
5. Collections, files, and safer code
6. NumPy arrays and numerical reasoning
7. pandas DataFrame manipulation
8. Matplotlib for business visualization
9. A compact end-to-end sales workflow
10. Responsible use, reproducibility, and handoff
11. Exercises

## Learning objectives

After completing this notebook, you should be able to read and modify basic Python code, choose appropriate built-in data structures, write a reusable function, handle common input errors, perform vectorized calculations with NumPy, inspect and clean a pandas DataFrame, aggregate and merge business data, create basic charts, save analytical outputs, and document a reproducible workflow.

## Connection map

Appendix A introduced Google Colab as the workspace. Appendix B introduces the Python language used throughout the book. Later chapters assume that you can identify variables, follow function calls, inspect array and DataFrame shapes, interpret filtering and aggregation steps, and rerun a notebook from the first cell.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform
import random
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

WORK_DIR = Path('appendix_b_workspace')
DATA_DIR = WORK_DIR / 'data'
OUT_DIR = WORK_DIR / 'outputs'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

print(f'Python: {platform.python_version()}')
print(f'Working directory: {Path.cwd().resolve()}')
print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Output directory: {OUT_DIR.resolve()}')

## Utility functions

These small helpers keep later sections readable. They display compact tables, save JSON records, calculate file checksums, capture expected errors without stopping the notebook, and save charts consistently.

In [ ]:
# ============================================================
# Utility functions
# ============================================================
def show_table(frame, rows=10):
    """Display a compact copy of a pandas DataFrame."""
    display(frame.head(rows).copy())


def save_json(payload, path):
    """Save a JSON-serializable object with readable indentation."""
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)


def sha256_file(path):
    """Return a SHA-256 checksum for a file."""
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(65536), b''):
            digest.update(block)
    return digest.hexdigest()


def capture_error(label, function):
    """Run a function and summarize an expected error."""
    try:
        result = function()
        return {'example': label, 'error_type': 'No error', 'message': '', 'result': result}
    except Exception as exc:
        return {
            'example': label,
            'error_type': type(exc).__name__,
            'message': str(exc),
            'result': None,
        }


def save_current_figure(filename):
    """Finish, save, display, and close the current Matplotlib figure."""
    path = OUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    return path

## B.1 Python as the working language of business analytics

A line of Python should be interpreted as part of an analytical chain. A business question determines the required evidence. Data preparation makes that evidence credible. Analysis and visualization reveal patterns. Interpretation connects those patterns to a decision while preserving uncertainty and limitations.

In [ ]:
# ============================================================
# B.1.1 The analytics chain and core Python ecosystem
# ============================================================
analytics_chain = pd.DataFrame([
    {'stage': 'Business question', 'example': 'Which regions have weak profit performance?', 'Python contribution': 'Store the decision contract and assumptions'},
    {'stage': 'Data preparation', 'example': 'Correct invalid numbers and inconsistent labels', 'Python contribution': 'Apply repeatable cleaning rules'},
    {'stage': 'Analysis', 'example': 'Calculate revenue, profit, and margin by region', 'Python contribution': 'Use NumPy and pandas operations'},
    {'stage': 'Communication', 'example': 'Show regional and monthly patterns', 'Python contribution': 'Create tables and Matplotlib charts'},
    {'stage': 'Decision support', 'example': 'Prioritize a region for investigation', 'Python contribution': 'Save evidence and document limitations'},
])

python_ecosystem = pd.DataFrame([
    {'analytical_need': 'Numerical operations', 'tool': 'NumPy', 'typical_use': 'Arrays, ratios, distances, and matrix calculations'},
    {'analytical_need': 'Structured data', 'tool': 'pandas', 'typical_use': 'Clean, join, filter, and aggregate business records'},
    {'analytical_need': 'Visualization', 'tool': 'Matplotlib', 'typical_use': 'Line, bar, histogram, and scatter charts'},
    {'analytical_need': 'Modeling', 'tool': 'scikit-learn and related libraries', 'typical_use': 'Regression, classification, clustering, and evaluation'},
    {'analytical_need': 'Interactive work', 'tool': 'Jupyter or Colab', 'typical_use': 'Combine explanation, code, output, and interpretation'},
])

show_table(analytics_chain)
show_table(python_ecosystem)

## B.2 Setting up a practical Python workspace

Colab is the fastest browser-based starting point. Jupyter is useful for local exploratory work. Anaconda simplifies local package management, and Visual Studio Code is useful when an analysis grows into multiple scripts and files. The best workspace is the simplest one that supports the task and can be rerun reliably.

In [ ]:
# ============================================================
# B.2.1 Workspace and import choices
# ============================================================
workspace_guide = pd.DataFrame([
    {'workspace': 'Google Colab', 'best_use': 'Learning and quick experiments', 'main_caution': 'Runtime files and state are temporary'},
    {'workspace': 'Jupyter Notebook', 'best_use': 'Local step-by-step exploration', 'main_caution': 'Hidden execution order can cause confusion'},
    {'workspace': 'Anaconda', 'best_use': 'Beginner-friendly local installation', 'main_caution': 'The installation is relatively large'},
    {'workspace': 'Visual Studio Code', 'best_use': 'Scripts, debugging, and multi-file projects', 'main_caution': 'It requires more setup'},
])

import_aliases = pd.DataFrame([
    {'statement': 'import numpy as np', 'meaning': 'Use np as a short name for NumPy'},
    {'statement': 'import pandas as pd', 'meaning': 'Use pd as a short name for pandas'},
    {'statement': 'import matplotlib.pyplot as plt', 'meaning': 'Use plt for plotting commands'},
])

show_table(workspace_guide)
show_table(import_aliases)

In [ ]:
# ============================================================
# B.2.2 Inspect the current Python environment
# ============================================================
package_names = ['numpy', 'pandas', 'matplotlib']
package_versions = []

for package in package_names:
    package_versions.append({
        'package': package,
        'version': importlib.metadata.version(package),
    })

environment_summary = pd.DataFrame([
    {'item': 'Python executable', 'value': sys.executable},
    {'item': 'Python version', 'value': platform.python_version()},
    {'item': 'Operating system', 'value': platform.platform()},
    {'item': 'Random seed', 'value': SEED},
])

show_table(environment_summary, rows=10)
show_table(pd.DataFrame(package_versions), rows=10)

print('Installation pattern for a missing package: %pip install package_name')
print('Keep required installation commands near the top of a Colab notebook.')

## B.3 Core syntax for reading and writing analytics code

Python syntax is readable but precise. Variables store values, operators transform or compare them, conditions select actions, loops repeat actions, and functions package reusable logic. The goal is not to memorize every rule. The goal is to recognize these recurring patterns in analytical code.

In [ ]:
# ============================================================
# B.3.1 Variables, basic types, and formatted output
# ============================================================
campaign_name = 'Spring Retention'
impressions = 12_500
click_rate = 0.041
is_active = True
launch_month = '2026-03'

variable_examples = pd.DataFrame([
    {'name': 'campaign_name', 'value': campaign_name, 'Python_type': type(campaign_name).__name__},
    {'name': 'impressions', 'value': impressions, 'Python_type': type(impressions).__name__},
    {'name': 'click_rate', 'value': click_rate, 'Python_type': type(click_rate).__name__},
    {'name': 'is_active', 'value': is_active, 'Python_type': type(is_active).__name__},
    {'name': 'launch_month', 'value': launch_month, 'Python_type': type(launch_month).__name__},
])
show_table(variable_examples)

print(f'{campaign_name} generated {impressions:,} impressions.')
print(f'Click rate: {click_rate:.1%}')
print(f'Active campaign: {is_active}')

In [ ]:
# ============================================================
# B.3.2 Arithmetic, comparison, and logical operators
# ============================================================
revenue = 125_000
cost = 91_000
profit = revenue - cost
margin = profit / revenue

operator_results = {
    'profit': profit,
    'margin': margin,
    'margin_at_least_20_percent': margin >= 0.20,
    'positive_profit_and_active': (profit > 0) and is_active,
    'needs_review': (margin < 0.20) or (not is_active),
}

for name, value in operator_results.items():
    print(f'{name}: {value}')

In [ ]:
# ============================================================
# B.3.3 Conditions connect evidence to a rule
# ============================================================
if margin >= 0.25:
    margin_status = 'strong'
elif margin >= 0.15:
    margin_status = 'acceptable'
else:
    margin_status = 'review'

print(f'Profit margin: {margin:.1%}')
print(f'Rule-based status: {margin_status}')

condition_guide = pd.DataFrame([
    {'condition': 'margin >= 0.25', 'action': 'Label strong'},
    {'condition': '0.15 <= margin < 0.25', 'action': 'Label acceptable'},
    {'condition': 'margin < 0.15', 'action': 'Send for review'},
])
show_table(condition_guide)

In [ ]:
# ============================================================
# B.3.4 Loops, zip, and enumerate
# ============================================================
months = ['Jan', 'Feb', 'Mar']
monthly_revenue = [100_000, 125_000, 118_000]
monthly_cost = [78_000, 91_000, 88_000]

monthly_rows = []
for position, (month, month_revenue, month_cost) in enumerate(
    zip(months, monthly_revenue, monthly_cost),
    start=1,
):
    monthly_rows.append({
        'position': position,
        'month': month,
        'revenue': month_revenue,
        'cost': month_cost,
        'profit': month_revenue - month_cost,
    })

monthly_performance = pd.DataFrame(monthly_rows)
show_table(monthly_performance)

In [ ]:
# ============================================================
# B.3.5 Functions package reusable business logic
# ============================================================
def profit_margin(revenue_value, cost_value):
    """Return profit margin, or None when revenue is zero."""
    if revenue_value == 0:
        return None
    return (revenue_value - cost_value) / revenue_value


def classify_margin(margin_value, strong_threshold=0.25, review_threshold=0.15):
    """Translate a numerical margin into a simple review label."""
    if margin_value is None:
        return 'undefined'
    if margin_value >= strong_threshold:
        return 'strong'
    if margin_value >= review_threshold:
        return 'acceptable'
    return 'review'

function_results = monthly_performance.copy()
function_results['margin'] = [
    profit_margin(r, c)
    for r, c in zip(function_results['revenue'], function_results['cost'])
]
function_results['status'] = function_results['margin'].apply(classify_margin)
show_table(function_results)

## B.4 Data structures, files, and safer code

Business data begins as collections of values. Lists preserve order and can change. Tuples usually represent fixed groups. Dictionaries map keys to values. Sets retain unique values. Files move data between sessions, while exception handling turns predictable failures into clear messages.

In [ ]:
# ============================================================
# B.4.1 Lists, tuples, dictionaries, and sets
# ============================================================
product_sequence = ['A100', 'B200', 'A100', 'C300']
analysis_period = ('2026-01-01', '2026-03-31')
product_prices = {'A100': 24.99, 'B200': 39.50, 'C300': 18.75}
unique_products = set(product_sequence)

structure_examples = pd.DataFrame([
    {'structure': 'list', 'example_value': str(product_sequence), 'business_use': 'Ordered product or monthly values'},
    {'structure': 'tuple', 'example_value': str(analysis_period), 'business_use': 'Fixed start and end dates'},
    {'structure': 'dict', 'example_value': str(product_prices), 'business_use': 'Map product code to price'},
    {'structure': 'set', 'example_value': str(sorted(unique_products)), 'business_use': 'Identify distinct products'},
])
show_table(structure_examples)

In [ ]:
# ============================================================
# B.4.2 Indexing, slicing, and comprehensions
# ============================================================
first_product = product_sequence[0]
last_product = product_sequence[-1]
first_three = product_sequence[:3]

sales_values = [1200, 0, 1500, -50, 1800]
positive_sales = [value for value in sales_values if value > 0]
discounted_prices = {
    product: round(price * 0.90, 2)
    for product, price in product_prices.items()
}

print('First product:', first_product)
print('Last product:', last_product)
print('First three entries:', first_three)
print('Positive sales:', positive_sales)
print('Discounted prices:', discounted_prices)

In [ ]:
# ============================================================
# B.4.3 Safe text and JSON file handling
# ============================================================
notes_path = DATA_DIR / 'analysis_notes.txt'
settings_path = DATA_DIR / 'campaign_settings.json'

notes_text = (
    'Business question: compare regional sales performance.\n'
    'Unit of analysis: one retail order.\n'
    'Important limitation: synthetic classroom data.\n'
)

with open(notes_path, 'w', encoding='utf-8') as handle:
    handle.write(notes_text)

campaign_settings = {
    'campaign': campaign_name,
    'minimum_margin': 0.15,
    'regions': ['West', 'East', 'Central', 'South'],
    'active': True,
}
save_json(campaign_settings, settings_path)

with open(notes_path, 'r', encoding='utf-8') as handle:
    loaded_notes = handle.read()
with open(settings_path, 'r', encoding='utf-8') as handle:
    loaded_settings = json.load(handle)

print(loaded_notes)
print(loaded_settings)

In [ ]:
# ============================================================
# B.4.4 Exception handling and defensive conversion
# ============================================================
def parse_currency(value):
    """Convert a value to float and return NaN when conversion fails."""
    try:
        return float(str(value).replace('$', '').replace(',', '').strip())
    except (TypeError, ValueError):
        return np.nan

currency_examples = ['$1,250.50', '980', 'not available', None]
parsed_currency = pd.DataFrame({
    'raw_value': currency_examples,
    'parsed_value': [parse_currency(value) for value in currency_examples],
})
show_table(parsed_currency)

expected_errors = pd.DataFrame([
    capture_error('Missing file', lambda: (DATA_DIR / 'missing.csv').read_text()),
    capture_error('Invalid integer', lambda: int('twelve')),
    capture_error('Missing dictionary key', lambda: product_prices['Z999']),
])
show_table(expected_errors)

## B.5 NumPy for numerical thinking

A NumPy array stores numerical values in a defined shape. A one-dimensional array can represent a time series. A two-dimensional array commonly uses rows for observations and columns for features. Vectorized operations apply a calculation to an entire array, and broadcasting applies a compatible smaller array across a larger one.

In [ ]:
# ============================================================
# B.5.1 Arrays, dimensions, shapes, and data types
# ============================================================
sales_array = np.array([1200, 1500, 1800, 2100], dtype=float)
feature_matrix = np.array([
    [12, 3, 0.10],
    [7, 5, 0.00],
    [21, 1, 0.15],
    [4, 8, 0.05],
], dtype=float)

array_summary = pd.DataFrame([
    {'object': 'sales_array', 'ndim': sales_array.ndim, 'shape': str(sales_array.shape), 'dtype': str(sales_array.dtype)},
    {'object': 'feature_matrix', 'ndim': feature_matrix.ndim, 'shape': str(feature_matrix.shape), 'dtype': str(feature_matrix.dtype)},
])
show_table(array_summary)
print('Feature matrix:\n', feature_matrix)

In [ ]:
# ============================================================
# B.5.2 Vectorization and Boolean indexing
# ============================================================
growth_rate = 0.08
vectorized_forecast = sales_array * (1 + growth_rate)
loop_forecast = np.array([value * (1 + growth_rate) for value in sales_array])

high_sales_mask = sales_array >= 1700
high_sales_values = sales_array[high_sales_mask]

print('Vectorized forecast:', vectorized_forecast)
print('Loop forecast:', loop_forecast)
print('Same numerical result:', np.allclose(vectorized_forecast, loop_forecast))
print('High-sales mask:', high_sales_mask)
print('High-sales values:', high_sales_values)

In [ ]:
# ============================================================
# B.5.3 Broadcasting and aggregation by axis
# ============================================================
store_month_sales = np.array([
    [100, 110, 120],
    [90, 105, 115],
    [130, 125, 140],
    [80, 95, 100],
], dtype=float)

month_adjustment = np.array([1.00, 1.05, 0.98])
adjusted_sales = store_month_sales * month_adjustment

store_totals = adjusted_sales.sum(axis=1)
month_averages = adjusted_sales.mean(axis=0)

print('Original matrix shape:', store_month_sales.shape)
print('Adjustment shape:', month_adjustment.shape)
print('Adjusted sales:\n', np.round(adjusted_sales, 2))
print('Store totals:', np.round(store_totals, 2))
print('Month averages:', np.round(month_averages, 2))

In [ ]:
# ============================================================
# B.5.4 Matrix multiplication as a weighted score
# ============================================================
# Columns represent recency, purchase frequency, and email engagement.
customer_features = np.array([
    [0.2, 0.8, 0.6],
    [0.9, 0.2, 0.3],
    [0.4, 0.6, 0.9],
])
feature_weights = np.array([-0.5, 0.8, 0.4])

customer_scores = customer_features @ feature_weights
score_table = pd.DataFrame({
    'customer': ['Customer A', 'Customer B', 'Customer C'],
    'weighted_score': customer_scores,
})
show_table(score_table)

print('Feature matrix shape:', customer_features.shape)
print('Weight vector shape:', feature_weights.shape)
print('Score vector shape:', customer_scores.shape)

## B.6 pandas for DataFrame manipulation

A pandas DataFrame resembles a spreadsheet, but each transformation is explicit and repeatable. A reliable workflow begins by inspecting shape, columns, data types, missing values, and duplicates. Cleaning rules should be fitted to the business meaning of each variable rather than applied mechanically.

In [ ]:
# ============================================================
# B.6.1 Generate synthetic source files with realistic quality issues
# ============================================================
n_orders = 480
regions = np.array(['West', 'East', 'Central', 'South'])
channels = np.array(['Store', 'Web', 'Marketplace'])
products = np.array(['A100', 'B200', 'C300', 'D400', 'E500'])
price_map = {'A100': 24.99, 'B200': 39.50, 'C300': 18.75, 'D400': 62.00, 'E500': 14.25}
cost_ratio_map = {'A100': 0.58, 'B200': 0.63, 'C300': 0.55, 'D400': 0.68, 'E500': 0.52}

product_choice = RNG.choice(products, size=n_orders, p=[0.24, 0.20, 0.22, 0.14, 0.20])
units = RNG.integers(1, 7, size=n_orders)
discount_rate = RNG.choice([0.00, 0.05, 0.10, 0.15, 0.20], size=n_orders, p=[0.28, 0.22, 0.24, 0.17, 0.09])
unit_price = np.array([price_map[product] for product in product_choice])
base_revenue = units * unit_price * (1 - discount_rate)
revenue_values = base_revenue * RNG.lognormal(mean=0.0, sigma=0.06, size=n_orders)
cost_values = np.array([cost_ratio_map[product] for product in product_choice]) * units * unit_price

dates = pd.Timestamp('2026-01-01') + pd.to_timedelta(RNG.integers(0, 181, size=n_orders), unit='D')

raw_sales = pd.DataFrame({
    'OrderID': [f'O{10000 + i}' for i in range(n_orders)],
    'OrderDate': dates.strftime('%Y-%m-%d'),
    'CustomerID': [f'C{1000 + value}' for value in RNG.integers(0, 125, size=n_orders)],
    'Region': RNG.choice(regions, size=n_orders, p=[0.31, 0.25, 0.23, 0.21]),
    'Channel': RNG.choice(channels, size=n_orders, p=[0.46, 0.39, 0.15]),
    'ProductCode': product_choice,
    'Units': units,
    'DiscountRate': discount_rate,
    'Revenue': [f'{value:.2f}' for value in revenue_values],
    'Cost': [f'{value:.2f}' for value in cost_values],
})

# Add several realistic data-quality problems.
raw_sales.loc[5, 'Region'] = ' west '
raw_sales.loc[18, 'Region'] = 'EAST'
raw_sales.loc[33, 'Region'] = None
raw_sales.loc[44, 'Revenue'] = 'not available'
raw_sales.loc[75, 'Cost'] = ''
raw_sales.loc[109, 'OrderDate'] = '2026-02-30'
raw_sales.loc[142, 'Units'] = 0
raw_sales = pd.concat([raw_sales, raw_sales.iloc[[10, 11]]], ignore_index=True)

product_lookup = pd.DataFrame([
    {'ProductCode': 'A100', 'ProductName': 'Everyday Bottle', 'Category': 'Home'},
    {'ProductCode': 'B200', 'ProductName': 'Travel Organizer', 'Category': 'Accessories'},
    {'ProductCode': 'C300', 'ProductName': 'Desk Notebook', 'Category': 'Office'},
    {'ProductCode': 'D400', 'ProductName': 'Portable Speaker', 'Category': 'Electronics'},
    {'ProductCode': 'E500', 'ProductName': 'Reusable Tote', 'Category': 'Accessories'},
])

raw_sales_path = DATA_DIR / 'raw_sales.csv'
product_lookup_path = DATA_DIR / 'product_lookup.csv'
raw_sales.to_csv(raw_sales_path, index=False)
product_lookup.to_csv(product_lookup_path, index=False)
raw_sales_checksum = sha256_file(raw_sales_path)

print(f'Raw sales file: {raw_sales_path}')
print(f'Product lookup file: {product_lookup_path}')
print(f'Raw rows written: {len(raw_sales):,}')
show_table(raw_sales)

In [ ]:
# ============================================================
# B.6.2 Load and inspect before transforming
# ============================================================
sales_loaded = pd.read_csv(raw_sales_path)

profile = pd.DataFrame({
    'column': sales_loaded.columns,
    'dtype': [str(dtype) for dtype in sales_loaded.dtypes],
    'missing_values': sales_loaded.isna().sum().values,
    'unique_values': sales_loaded.nunique(dropna=True).values,
})

print('Shape:', sales_loaded.shape)
print('Exact duplicate rows:', int(sales_loaded.duplicated().sum()))
show_table(sales_loaded)
show_table(profile, rows=20)

In [ ]:
# ============================================================
# B.6.3 Select columns, filter rows, and sort records
# ============================================================
selected_columns = sales_loaded[['OrderID', 'OrderDate', 'Region', 'Channel', 'Revenue']]
web_orders = sales_loaded.loc[
    (sales_loaded['Channel'] == 'Web') & (sales_loaded['Units'] >= 3),
    ['OrderID', 'Region', 'ProductCode', 'Units', 'Revenue'],
].sort_values(['Units', 'OrderID'], ascending=[False, True])

print('Selected column shape:', selected_columns.shape)
print('Filtered Web order count:', len(web_orders))
show_table(web_orders)

In [ ]:
# ============================================================
# B.6.4 Clean types, labels, missing values, and duplicates
# ============================================================
def clean_sales_data(frame):
    """Return an analysis-ready sales table and a compact cleaning audit."""
    cleaned = frame.copy()
    initial_rows = len(cleaned)
    exact_duplicates = int(cleaned.duplicated().sum())

    cleaned.columns = cleaned.columns.str.strip()
    cleaned['Region'] = cleaned['Region'].astype('string').str.strip().str.title()
    cleaned['Channel'] = cleaned['Channel'].astype('string').str.strip().str.title()
    cleaned['OrderDate'] = pd.to_datetime(cleaned['OrderDate'], errors='coerce')

    numeric_columns = ['Units', 'DiscountRate', 'Revenue', 'Cost']
    for column in numeric_columns:
        cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')

    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.drop_duplicates(subset=['OrderID'], keep='first')
    cleaned = cleaned.dropna(subset=[
        'OrderID', 'OrderDate', 'CustomerID', 'Region', 'Channel',
        'ProductCode', 'Units', 'Revenue', 'Cost',
    ])
    cleaned = cleaned.loc[
        cleaned['Region'].isin(['West', 'East', 'Central', 'South'])
        & (cleaned['Units'] > 0)
        & (cleaned['Revenue'] > 0)
        & (cleaned['Cost'] >= 0)
    ].copy()

    cleaned['Profit'] = cleaned['Revenue'] - cleaned['Cost']
    cleaned['Margin'] = cleaned['Profit'] / cleaned['Revenue']
    cleaned['Month'] = cleaned['OrderDate'].dt.to_period('M').astype(str)
    cleaned = cleaned.sort_values(['OrderDate', 'OrderID']).reset_index(drop=True)

    audit = {
        'initial_rows': int(initial_rows),
        'exact_duplicate_rows_detected': exact_duplicates,
        'final_rows': int(len(cleaned)),
        'rows_removed': int(initial_rows - len(cleaned)),
        'remaining_missing_cells': int(cleaned.isna().sum().sum()),
        'unique_order_ids': int(cleaned['OrderID'].nunique()),
    }
    return cleaned, audit

cleaned_sales, cleaning_audit = clean_sales_data(sales_loaded)
show_table(pd.DataFrame([cleaning_audit]))
show_table(cleaned_sales)
print(cleaned_sales.dtypes)

In [ ]:
# ============================================================
# B.6.5 Group, aggregate, and reshape
# ============================================================
region_summary = (
    cleaned_sales.groupby('Region', as_index=False)
    .agg(
        orders=('OrderID', 'nunique'),
        customers=('CustomerID', 'nunique'),
        total_revenue=('Revenue', 'sum'),
        total_profit=('Profit', 'sum'),
        average_order_value=('Revenue', 'mean'),
    )
)
region_summary['profit_margin'] = region_summary['total_profit'] / region_summary['total_revenue']
region_summary = region_summary.sort_values('total_revenue', ascending=False).reset_index(drop=True)

monthly_region_pivot = pd.pivot_table(
    cleaned_sales,
    index='Month',
    columns='Region',
    values='Revenue',
    aggfunc='sum',
    fill_value=0,
)

show_table(region_summary)
display(monthly_region_pivot)

In [ ]:
# ============================================================
# B.6.6 Merge related tables using a shared key
# ============================================================
enriched_sales = cleaned_sales.merge(
    product_lookup,
    on='ProductCode',
    how='left',
    validate='many_to_one',
    indicator=True,
)

category_summary = (
    enriched_sales.groupby('Category', as_index=False)
    .agg(
        orders=('OrderID', 'nunique'),
        total_revenue=('Revenue', 'sum'),
        total_profit=('Profit', 'sum'),
    )
    .sort_values('total_revenue', ascending=False)
)
category_summary['profit_margin'] = category_summary['total_profit'] / category_summary['total_revenue']

print(enriched_sales['_merge'].value_counts())
show_table(enriched_sales[['OrderID', 'ProductCode', 'ProductName', 'Category', 'Revenue', 'Profit']])
show_table(category_summary)

## B.7 Matplotlib for basic business visualization

A chart should answer a question more clearly than raw rows. Use a line chart for change over time, a bar chart for category comparison, a histogram for a distribution, and a scatter plot for the relationship between two numerical variables. Titles and axis labels should state what the reader is seeing.

In [ ]:
# ============================================================
# B.7.1 Match the chart to the analytical question
# ============================================================
chart_choice = pd.DataFrame([
    {'question': 'How does revenue change over time?', 'chart': 'Line chart', 'main_visual_task': 'Follow a trend'},
    {'question': 'Which region has the highest revenue?', 'chart': 'Bar chart', 'main_visual_task': 'Compare categories'},
    {'question': 'How dispersed are order values?', 'chart': 'Histogram', 'main_visual_task': 'Inspect a distribution'},
    {'question': 'Do discount and revenue move together?', 'chart': 'Scatter plot', 'main_visual_task': 'Inspect an association'},
])
show_table(chart_choice)

In [ ]:
# ============================================================
# B.7.2 Line chart: monthly revenue
# ============================================================
monthly_revenue_summary = (
    cleaned_sales.groupby('Month', as_index=False)['Revenue']
    .sum()
    .sort_values('Month')
)

plt.figure(figsize=(8, 4.5))
plt.plot(monthly_revenue_summary['Month'], monthly_revenue_summary['Revenue'])
plt.title('Monthly Revenue')
plt.xlabel('Month')
plt.ylabel('Total revenue')
plt.xticks(rotation=45)

monthly_line_path = save_current_figure('monthly_revenue_line.png')
print(f'Saved to: {monthly_line_path}')

In [ ]:
# ============================================================
# B.7.3 Bar chart: revenue by region
# ============================================================
region_for_plot = region_summary.sort_values('total_revenue')

plt.figure(figsize=(7, 4.5))
plt.bar(region_for_plot['Region'], region_for_plot['total_revenue'])
plt.title('Total Revenue by Region')
plt.xlabel('Region')
plt.ylabel('Total revenue')

region_bar_path = save_current_figure('regional_revenue_bar.png')
print(f'Saved to: {region_bar_path}')

In [ ]:
# ============================================================
# B.7.4 Histogram: order revenue distribution
# ============================================================
plt.figure(figsize=(7, 4.5))
plt.hist(cleaned_sales['Revenue'], bins=20)
plt.title('Distribution of Order Revenue')
plt.xlabel('Order revenue')
plt.ylabel('Number of orders')

revenue_hist_path = save_current_figure('order_revenue_histogram.png')
print(f'Saved to: {revenue_hist_path}')

In [ ]:
# ============================================================
# B.7.5 Scatter plot: discount and revenue
# ============================================================
plt.figure(figsize=(7, 4.5))
plt.scatter(cleaned_sales['DiscountRate'], cleaned_sales['Revenue'])
plt.title('Discount Rate and Order Revenue')
plt.xlabel('Discount rate')
plt.ylabel('Order revenue')

discount_scatter_path = save_current_figure('discount_revenue_scatter.png')
print(f'Saved to: {discount_scatter_path}')

In [ ]:
# ============================================================
# B.7.6 Interpret the evidence without making a causal claim
# ============================================================
best_revenue_region = region_summary.loc[region_summary['total_revenue'].idxmax(), 'Region']
best_margin_region = region_summary.loc[region_summary['profit_margin'].idxmax(), 'Region']
discount_revenue_correlation = cleaned_sales[['DiscountRate', 'Revenue']].corr().iloc[0, 1]

interpretation = pd.DataFrame([
    {'finding': 'Highest total revenue region', 'value': best_revenue_region, 'caution': 'Revenue does not measure profitability by itself'},
    {'finding': 'Highest aggregate profit margin region', 'value': best_margin_region, 'caution': 'The result reflects this synthetic period only'},
    {'finding': 'Discount-revenue correlation', 'value': f'{discount_revenue_correlation:.3f}', 'caution': 'Correlation is not a causal effect of discounting'},
])
show_table(interpretation)

## B.8 A compact sales analysis workflow

A reusable workflow separates source data, transformation logic, analytical outputs, and interpretation. The business question comes first. The function below loads the two source files, applies documented cleaning rules, validates the product join, calculates summaries, saves results, and returns a compact audit record.

In [ ]:
# ============================================================
# B.8.1 State the decision contract before analysis
# ============================================================
decision_contract = {
    'business_question': 'Which regions and product categories warrant a profit-performance review?',
    'unit_of_analysis': 'One completed retail order',
    'analysis_period': 'January through June 2026',
    'primary_metrics': ['total_revenue', 'total_profit', 'profit_margin'],
    'decision_user': 'Retail performance manager',
    'important_caution': 'Synthetic data demonstrates workflow mechanics and supports no real business action.',
}

show_table(pd.DataFrame([
    {'field': key, 'value': value if isinstance(value, str) else ', '.join(value)}
    for key, value in decision_contract.items()
]))

In [ ]:
# ============================================================
# B.8.2 Build and run a reusable end-to-end workflow
# ============================================================
def run_sales_workflow(raw_path, lookup_path, output_dir):
    """Load, clean, enrich, summarize, validate, and save a sales analysis."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_frame = pd.read_csv(raw_path)
    lookup_frame = pd.read_csv(lookup_path)
    clean_frame, audit = clean_sales_data(raw_frame)

    enriched_frame = clean_frame.merge(
        lookup_frame,
        on='ProductCode',
        how='left',
        validate='many_to_one',
    )

    if enriched_frame['Category'].isna().any():
        raise ValueError('At least one product did not match the product lookup table.')

    regional = (
        enriched_frame.groupby('Region', as_index=False)
        .agg(
            orders=('OrderID', 'nunique'),
            total_revenue=('Revenue', 'sum'),
            total_profit=('Profit', 'sum'),
        )
    )
    regional['profit_margin'] = regional['total_profit'] / regional['total_revenue']
    regional = regional.sort_values('total_revenue', ascending=False).reset_index(drop=True)

    category = (
        enriched_frame.groupby('Category', as_index=False)
        .agg(
            orders=('OrderID', 'nunique'),
            total_revenue=('Revenue', 'sum'),
            total_profit=('Profit', 'sum'),
        )
    )
    category['profit_margin'] = category['total_profit'] / category['total_revenue']
    category = category.sort_values('total_revenue', ascending=False).reset_index(drop=True)

    monthly = (
        enriched_frame.groupby('Month', as_index=False)
        .agg(total_revenue=('Revenue', 'sum'), total_profit=('Profit', 'sum'))
        .sort_values('Month')
    )

    assert enriched_frame['OrderID'].is_unique
    assert enriched_frame['Revenue'].gt(0).all()
    assert enriched_frame['Cost'].ge(0).all()
    assert enriched_frame.isna().sum().sum() == 0

    data_path = output_dir / 'analysis_ready_sales.csv'
    regional_path = output_dir / 'regional_summary.csv'
    category_path = output_dir / 'category_summary.csv'
    monthly_path = output_dir / 'monthly_summary.csv'
    audit_path = output_dir / 'workflow_audit.json'

    enriched_frame.to_csv(data_path, index=False)
    regional.to_csv(regional_path, index=False)
    category.to_csv(category_path, index=False)
    monthly.to_csv(monthly_path, index=False)

    audit_record = {
        **audit,
        'raw_file_checksum': sha256_file(raw_path),
        'lookup_rows': int(len(lookup_frame)),
        'final_columns': list(enriched_frame.columns),
        'completed_utc': datetime.now(timezone.utc).isoformat(),
    }
    save_json(audit_record, audit_path)

    return {
        'analysis_ready': enriched_frame,
        'regional_summary': regional,
        'category_summary': category,
        'monthly_summary': monthly,
        'audit': audit_record,
        'saved_paths': [data_path, regional_path, category_path, monthly_path, audit_path],
    }

workflow_result = run_sales_workflow(raw_sales_path, product_lookup_path, OUT_DIR)
show_table(workflow_result['regional_summary'])
show_table(workflow_result['category_summary'])
show_table(pd.DataFrame([workflow_result['audit']]))

In [ ]:
# ============================================================
# B.8.3 Inspect saved outputs and verify the source file
# ============================================================
manifest_rows = []
for path in sorted(OUT_DIR.glob('*')):
    if path.is_file():
        manifest_rows.append({
            'file': path.name,
            'size_bytes': path.stat().st_size,
            'sha256_prefix': sha256_file(path)[:16],
        })

output_manifest = pd.DataFrame(manifest_rows)
raw_source_unchanged = sha256_file(raw_sales_path) == raw_sales_checksum

show_table(output_manifest, rows=30)
print('Raw source file unchanged:', raw_source_unchanged)
print('Final analysis rows:', len(workflow_result['analysis_ready']))

## B.9 Responsible use of Python in business analytics

Correct syntax is not the same as reliable analysis. Keep raw data separate from transformed data, document why rows are removed, validate joins, inspect row counts before and after cleaning, avoid hard-coded credentials, and distinguish observed associations from causal claims. A notebook should be understandable by a reviewer who did not write the code.

In [ ]:
# ============================================================
# B.9.1 Quality, privacy, and lineage checks
# ============================================================
def privacy_column_audit(columns):
    """Flag column names that may contain identifiers or sensitive information."""
    sensitive_terms = ['email', 'phone', 'address', 'ssn', 'password', 'token', 'secret', 'api_key']
    identifier_terms = ['customerid', 'customer_id', 'userid', 'user_id', 'accountid', 'deviceid']
    rows = []
    for column in columns:
        normalized = column.lower().replace(' ', '')
        if any(term in normalized for term in sensitive_terms):
            risk = 'Sensitive or credential-like'
            action = 'Remove, mask, or use an approved secure process'
        elif any(term in normalized for term in identifier_terms):
            risk = 'Identifier'
            action = 'Confirm that row-level identity is necessary'
        else:
            risk = 'No keyword flag'
            action = 'Still inspect values and business context'
        rows.append({'column': column, 'risk_flag': risk, 'recommended_action': action})
    return pd.DataFrame(rows)

lineage = pd.DataFrame([
    {'stage': 'Raw source', 'artifact': raw_sales_path.name, 'control': 'Checksum recorded and file left unchanged'},
    {'stage': 'Cleaning', 'artifact': 'clean_sales_data()', 'control': 'Rules are explicit and row removal is audited'},
    {'stage': 'Enrichment', 'artifact': product_lookup_path.name, 'control': 'Many-to-one join is validated'},
    {'stage': 'Analysis', 'artifact': 'regional, category, and monthly summaries', 'control': 'Metrics have defined business meaning'},
    {'stage': 'Handoff', 'artifact': 'CSV and JSON outputs', 'control': 'Files include checksums and limitations'},
])

show_table(lineage)
show_table(privacy_column_audit(workflow_result['analysis_ready'].columns), rows=30)

In [ ]:
# ============================================================
# B.9.2 Save a notebook handoff card
# ============================================================
handoff_card = {
    'notebook_name': 'Appendix_B_Python_Basics.ipynb',
    'purpose': 'Beginner practice with Python, NumPy, pandas, Matplotlib, and reproducible business analysis',
    'data_source': 'Synthetic retail orders generated inside the notebook',
    'unit_of_analysis': decision_contract['unit_of_analysis'],
    'analysis_period': decision_contract['analysis_period'],
    'random_seed': SEED,
    'source_files': [raw_sales_path.name, product_lookup_path.name],
    'source_checksum': raw_sales_checksum,
    'primary_outputs': [path.name for path in workflow_result['saved_paths']],
    'known_limitations': [
        'The data is synthetic and does not represent a real company.',
        'The workflow is instructional and is not approved for deployment.',
        'Observed correlations should not be interpreted as causal effects.',
        'Keyword-based privacy checks cannot replace human review.',
    ],
    'created_utc': datetime.now(timezone.utc).isoformat(),
}

handoff_path = OUT_DIR / 'notebook_handoff_card.json'
save_json(handoff_card, handoff_path)

readiness_checklist = pd.DataFrame([
    {'item': 'Business question and unit of analysis are stated', 'status': 'complete'},
    {'item': 'Raw data is separate from transformed outputs', 'status': 'complete'},
    {'item': 'Data types, missing values, and duplicates are inspected', 'status': 'complete'},
    {'item': 'Cleaning rules and row removals are documented', 'status': 'complete'},
    {'item': 'Join cardinality and unmatched records are checked', 'status': 'complete'},
    {'item': 'Charts have titles and axis labels', 'status': 'complete'},
    {'item': 'Source checksum and output manifest are available', 'status': 'complete'},
    {'item': 'Notebook can run from a clean runtime', 'status': 'verify before sharing'},
])

show_table(readiness_checklist, rows=20)
print(f'Handoff card saved to: {handoff_path}')

## Decision guide

Use ordinary Python variables and collections for small pieces of logic. Use NumPy when repeated numerical operations and array shapes matter. Use pandas when the unit of analysis is organized as rows and variables as columns. Use Matplotlib when a chart clarifies a specific comparison. Keep raw inputs unchanged, write transformations as code, inspect every intermediate result, and save both evidence and documentation for the next analyst.

## Exercises

1. Create variables for campaign spend, attributed revenue, and number of customers. Print return on ad spend and revenue per customer with formatted output.

2. Modify the margin thresholds in `classify_margin()`. Explain how the status distribution changes.

3. Add April data to `monthly_performance` by extending the lists before the loop. Confirm that the function-based margin calculation still works.

4. Create a dictionary that maps each channel to a target conversion rate. Use a set to list the unique target values.

5. Add `'$2,400.00'` and `'N/A'` to `currency_examples`. Confirm that `parse_currency()` handles both values safely.

6. Create a 3 by 4 NumPy matrix of weekly sales. Calculate row totals, column averages, and values above a threshold.

7. Introduce one additional inconsistent region label into `sales_loaded`. Update `clean_sales_data()` only if the existing rule does not correct it.

8. Calculate regional revenue by channel with `groupby()`. Reshape the result into a pivot table.

9. Add a new product to the raw sales file but not to the lookup table. Rerun the workflow and explain why the validation should fail.

10. Create a chart of monthly profit. Save it to the output directory and add it to the output manifest.

11. Add a `CustomerEmail` column to a copy of the analysis-ready data. Run the privacy audit, then remove the column before saving.

12. Restart the runtime and run all cells. Record any failure caused by hidden state and correct the workflow rather than recreating the missing object manually.

In [ ]:
# Optional exercise starter: regional revenue by channel.
exercise_summary = (
    cleaned_sales.groupby(['Region', 'Channel'], as_index=False)
    .agg(total_revenue=('Revenue', 'sum'), orders=('OrderID', 'nunique'))
    .sort_values(['Region', 'total_revenue'], ascending=[True, False])
)

exercise_pivot = exercise_summary.pivot(
    index='Region',
    columns='Channel',
    values='total_revenue',
).fillna(0)

show_table(exercise_summary, rows=20)
display(exercise_pivot)